# Examen Aplicado — Machine Learning I

**Predicción del volumen de tráfico horario en la autopista Interestatal 94 (Minneapolis, EE.UU.)**

Esteban Molina Almazabal · Ingeniería en Inteligencia Artificial · Universidad Mayor

---

Este notebook desarrolla un problema de **regresión** de punta a punta: exploración,
limpieza, reducción de dimensionalidad, segmentación no supervisada y modelamiento
supervisado con validación cruzada.

Cada sección está rotulada con el número del punto de la ficha que responde, para
facilitar la revisión.

## Punto 3 · Declaración del dataset

| Campo | Valor |
|---|---|
| **Nombre** | Metro Interstate Traffic Volume |
| **Fuente** | UCI Machine Learning Repository (dataset 492) |
| **URL** | https://archive.ics.uci.edu/dataset/492/metro+interstate+traffic+volume |
| **Descarga directa** | https://archive.ics.uci.edu/static/public/492/metro+interstate+traffic+volume.zip |
| **Filas (original)** | 48.204 |
| **Columnas (original)** | 9 |
| **Variable objetivo** | `traffic_volume` — número de vehículos por hora |
| **Tipo de tarea** | **Regresión** (variable objetivo continua de conteo) |
| **Licencia** | Creative Commons Attribution 4.0 International (CC BY 4.0) |

**Contexto.** Los datos provienen de la estación de conteo ATR 301 de la autopista
I-94, en dirección oeste, entre Minneapolis y St. Paul. Cada fila es una hora e
informa cuántos vehículos pasaron, junto con las condiciones meteorológicas de esa
hora y si correspondía a un feriado. El período cubierto va de octubre de 2012 a
septiembre de 2018.

**Por qué este dataset.** Cumple holgadamente los mínimos exigidos (≥500 observaciones,
≥6 predictoras, ≥3 numéricas continuas) y no pertenece a la lista de datasets
excluidos. Además presenta problemas de calidad de datos genuinos —duplicados,
valores físicamente imposibles y una categoría mal codificada— que permiten justificar
decisiones de preprocesamiento con evidencia y no por trámite.

## Configuración del entorno

In [ ]:
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import (silhouette_score, mean_squared_error,
                             mean_absolute_error, r2_score)

warnings.filterwarnings("ignore")

RANDOM_STATE = 42          # semilla unica para todo el notebook (reproducibilidad)
np.random.seed(RANDOM_STATE)

FIGS = Path("figures")
FIGS.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 150
plt.rcParams["figure.autolayout"] = True

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

def guardar(nombre):
    '''Guarda la figura activa en figures/ a 150 dpi.'''
    plt.savefig(FIGS / nombre, dpi=150, bbox_inches="tight")

print("numpy      ", np.__version__)
print("pandas     ", pd.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)
print("semilla global:", RANDOM_STATE)

## Punto 4 · Carga del dataset e inspección inicial

Una advertencia sobre la lectura del archivo. La columna `holiday` usa el texto
literal `"None"` para las horas que no son feriado. Si se carga con las opciones
por omisión, pandas interpreta ese texto como valor faltante y aparecen **48.143
nulos falsos**. Cargamos primero de la forma ingenua para dejar el problema a la
vista en el punto 5, y luego de la forma correcta.

In [ ]:
RUTA = Path("data/raw/Metro_Interstate_Traffic_Volume.csv")

# --- lectura INGENUA (por omision): "None" se convierte en NaN ---
df_ingenuo = pd.read_csv(RUTA)

# --- lectura CORRECTA: "None" se conserva como texto ---
df = pd.read_csv(RUTA, keep_default_na=False, na_values=[""])

print("Dimensiones (df.shape):", df.shape)
print("\nNulos que reporta la lectura ingenua:", int(df_ingenuo.isna().sum().sum()))
print("Nulos que reporta la lectura correcta:", int(df.isna().sum().sum()))

In [ ]:
print("=== df.dtypes ===")
print(df.dtypes)

In [ ]:
print("=== df.info() ===")
df.info()

In [ ]:
print("=== df.describe() ===")
df.describe().T

In [ ]:
print("=== df.head() ===")
df.head()

## Punto 5 · Análisis de valores faltantes

Comparamos las dos lecturas para mostrar de dónde salen los nulos.

In [ ]:
pct_ingenuo = (df_ingenuo.isna().mean() * 100).round(2)
pct_real = (df.isna().mean() * 100).round(2)

comparacion = pd.DataFrame({
    "% nulos (lectura ingenua)": pct_ingenuo,
    "% nulos (lectura correcta)": pct_real,
})
comparacion

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(df_ingenuo.isna(), cbar=False, yticklabels=False,
            cmap=["#e8e8e8", "#c0392b"], ax=axes[0])
axes[0].set_title("Lectura ingenua: 'holiday' aparece 99,87% nula", fontsize=11)
axes[0].set_xlabel("Variable"); axes[0].set_ylabel("Observaciones")

sns.heatmap(df.isna(), cbar=False, yticklabels=False,
            cmap=["#e8e8e8", "#c0392b"], ax=axes[1])
axes[1].set_title("Lectura correcta: no hay valores faltantes", fontsize=11)
axes[1].set_xlabel("Variable"); axes[1].set_ylabel("Observaciones")

fig.suptitle("Mapa de calor de valores faltantes — Metro Interstate Traffic Volume",
             fontsize=13, y=1.03)
guardar("fig01_heatmap_nulos.png")
plt.show()

In [ ]:
print("Valores reales de 'holiday':")
print(df["holiday"].value_counts().to_string())
print("\nHoras que NO son feriado:", int((df['holiday'] == 'None').sum()))
print("Horas marcadas como feriado:", int((df['holiday'] != 'None').sum()))

### Decisión sobre valores faltantes — justificación

**El dataset no tiene valores faltantes.** Los 48.143 nulos de la lectura ingenua son
un artefacto: `"None"` es una categoría legítima que significa "esta hora no es
feriado", y pandas la confunde con un marcador de ausencia. **No se imputa ni se
elimina nada**, porque no hay nada que imputar; lo que corresponde es leer el archivo
con `keep_default_na=False`.

Sí hay un problema distinto en `holiday`, que se ve al contar sus valores: solo **61
de 48.204 horas** están marcadas como feriado. Un feriado dura 24 horas, así que
debería haber unas 1.400. La razón es que el dataset marca el feriado **únicamente en
el registro de las 00:00** de ese día y deja el resto de las horas como `"None"`. La
variable, tal como viene, es engañosa.

Decisión: colapsar `holiday` a una binaria `is_holiday` y, en el punto 7, reconstruir
la marca para las 24 horas del día feriado a partir de la fecha. Se conserva la
información sin arrastrar la codificación defectuosa.

## Limpieza estructural previa

Antes de tratar outliers hay dos defectos que corregir: timestamps duplicados y
niveles de texto inconsistentes.

In [ ]:
df["date_time"] = pd.to_datetime(df["date_time"])

print("Filas:", len(df))
print("Timestamps unicos:", df["date_time"].nunique())
print("Timestamps duplicados:", int(df["date_time"].duplicated().sum()))

ejemplo = df[df["date_time"].duplicated(keep=False)].head(6)
print("\nEjemplo de duplicados (misma hora, distinta descripcion del clima):")
print(ejemplo[["date_time", "weather_main", "weather_description", "traffic_volume"]].to_string(index=False))

In [ ]:
antes = len(df)
df = (df.drop_duplicates(subset="date_time", keep="first")
        .sort_values("date_time")
        .reset_index(drop=True))
print(f"Deduplicacion por date_time: {antes} -> {len(df)} filas (se eliminan {antes - len(df)})")

# normalizacion de texto: 'sky is clear' y 'Sky is Clear' eran niveles distintos
n_antes = df["weather_description"].nunique()
df["weather_description"] = df["weather_description"].str.lower().str.strip()
print(f"weather_description: {n_antes} -> {df['weather_description'].nunique()} niveles tras normalizar")
print("weather_main:", df["weather_main"].nunique(), "niveles")

**Justificación de la deduplicación.** 7.629 horas aparecen repetidas. Al inspeccionarlas
se ve que la hora y el volumen de tráfico coinciden, y lo que cambia es la descripción
del clima: cuando en esa hora hubo simultáneamente, por ejemplo, niebla y llovizna, la
fuente escribió una fila por condición. Es una duplicación de la dimensión meteorológica,
no una medición nueva de tráfico.

Mantenerlas sería un error grave: la misma hora entraría varias veces al entrenamiento,
y —peor— podría quedar repartida entre `train` y `test`, filtrando el valor objetivo de
test hacia el entrenamiento. Se conserva el primer registro de cada hora. Quedan 40.575
observaciones, muy por encima del mínimo de 500 exigido.

## Punto 6 · Detección de outliers por el método IQR

In [ ]:
NUM_CRUDAS = ["temp", "rain_1h", "snow_1h", "clouds_all", "traffic_volume"]

def resumen_iqr(data, cols):
    filas = []
    for c in cols:
        q1, q3 = data[c].quantile([0.25, 0.75])
        iqr = q3 - q1
        li, ls = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        fuera = ((data[c] < li) | (data[c] > ls)).sum()
        filas.append({"variable": c, "Q1": q1, "Q3": q3, "IQR": iqr,
                      "limite_inf": li, "limite_sup": ls,
                      "n_outliers": int(fuera),
                      "%_outliers": round(100 * fuera / len(data), 2),
                      "min": data[c].min(), "max": data[c].max()})
    return pd.DataFrame(filas)

tabla_iqr = resumen_iqr(df, NUM_CRUDAS)
tabla_iqr.round(3)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, c in zip(axes, ["temp", "rain_1h", "snow_1h", "clouds_all"]):
    sns.boxplot(y=df[c], ax=ax, color="#5b8ff9")
    ax.set_title(f"{c}\n(antes del tratamiento)", fontsize=10)
    ax.set_ylabel(c)
fig.suptitle("Boxplots ANTES del tratamiento de outliers", fontsize=13, y=1.05)
guardar("fig02_boxplots_antes.png")
plt.show()

print("Casos fisicamente imposibles:")
print("  temp = 0 Kelvin (cero absoluto):", int((df['temp'] < 200).sum()), "filas")
print("  rain_1h > 500 mm en una hora   :", int((df['rain_1h'] > 500).sum()), "filas")
print("\nLos 3 valores mas altos de rain_1h:")
print(df.nlargest(3, "rain_1h")[["date_time", "rain_1h", "weather_main", "traffic_volume"]].to_string(index=False))

### Decisión sobre outliers — justificación

El criterio IQR marca muchos valores como atípicos, pero **hay que separar dos cosas
distintas**: valores extremos que son reales y valores que son imposibles.

- `rain_1h` y `snow_1h` son cero el 99% del tiempo, así que el IQR marca como atípica
  **cualquier** precipitación. Eso es un artefacto de una distribución con exceso de
  ceros, no un error: que llueva 5 mm es un dato perfectamente válido. **Se mantienen.**
- `clouds_all` va de 0 a 100 por definición (es un porcentaje). **Se mantiene.**
- `traffic_volume` es la variable objetivo; su rango 0–7.280 es plausible para una
  autopista interestatal. **Se mantiene.** Eliminar outliers del target sesgaría el
  modelo hacia las horas cómodas.
- **`temp = 0 K` (10 filas)** es el cero absoluto, −273,15 °C. Físicamente imposible en
  Minneapolis: es una falla del sensor.
- **`rain_1h = 9.831,3 mm` (1 fila)** equivale a casi 10 metros de lluvia en una hora.
  El récord mundial en una hora es de unos 305 mm. Es un error de registro.

Solo estos dos últimos casos, **11 filas de 40.575 (0,03%)**, se marcan como `NaN`. No
se eliminan las filas —el resto de sus variables es válido y el tráfico de esas horas es
información útil— sino que se dejan que los impute el `SimpleImputer` **dentro del
pipeline**, es decir, usando la mediana calculada solo con datos de entrenamiento. Esa
es la diferencia entre imputar bien e imputar filtrando información del test.

In [ ]:
df.loc[df["temp"] < 200, "temp"] = np.nan
df.loc[df["rain_1h"] > 500, "rain_1h"] = np.nan
print("NaN introducidos a proposito -> temp:", int(df['temp'].isna().sum()),
      "| rain_1h:", int(df['rain_1h'].isna().sum()))

# Kelvin -> Celsius, para que los coeficientes e importancias sean interpretables
df["temp"] = df["temp"] - 273.15
print("\ntemp convertida a grados Celsius:")
print(df["temp"].describe().round(2).to_string())

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, c in zip(axes, ["temp", "rain_1h", "snow_1h", "clouds_all"]):
    sns.boxplot(y=df[c], ax=ax, color="#5ad8a6")
    ax.set_title(f"{c}\n(despues del tratamiento)", fontsize=10)
    ax.set_ylabel(c)
fig.suptitle("Boxplots DESPUÉS del tratamiento — temp en °C, imposibles marcados como NaN",
             fontsize=13, y=1.05)
guardar("fig03_boxplots_despues.png")
plt.show()

## Punto 7 · Distribución de la variable objetivo y asimetría

Al ser una tarea de regresión, corresponde graficar histograma con KDE y calcular la
asimetría para decidir si aplicar una transformación logarítmica.

In [ ]:
y_all = df["traffic_volume"]
skew = stats.skew(y_all)
kurt = stats.kurtosis(y_all)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
sns.histplot(y_all, kde=True, bins=60, color="#5b8ff9", ax=axes[0])
axes[0].set_title(f"Distribución de traffic_volume\nskewness = {skew:.4f} | curtosis = {kurt:.4f}")
axes[0].set_xlabel("Vehículos por hora"); axes[0].set_ylabel("Frecuencia")

sns.histplot(np.log1p(y_all), kde=True, bins=60, color="#e8684a", ax=axes[1])
axes[1].set_title(f"log1p(traffic_volume) — solo comparativo\nskewness = {stats.skew(np.log1p(y_all)):.4f}")
axes[1].set_xlabel("log(1 + vehículos por hora)"); axes[1].set_ylabel("Frecuencia")

fig.suptitle("Punto 7 — Variable objetivo: la distribución es bimodal, no asimétrica",
             fontsize=13, y=1.04)
guardar("fig04_target_hist_kde.png")
plt.show()

print(f"skewness = {skew:.4f}   |   curtosis = {kurt:.4f}")
print(f"skewness de log1p(y) = {stats.skew(np.log1p(y_all)):.4f}")

### Decisión sobre la transformación logarítmica

**No se transforma.** La regla de la ficha activa la transformación si la asimetría
supera 1,0, y aquí vale **−0,107**: la distribución es prácticamente simétrica. Aplicar
un logaritmo no solo sería innecesario, sino contraproducente — como se ve en el panel
derecho, `log1p` empeora la asimetría hasta **−1,49**, porque comprime la cola alta que
concentra la mayor parte de los datos.

Lo interesante está en la **curtosis de −1,30**, muy por debajo de 0. Junto con la forma
del histograma revela que la distribución es **bimodal**: hay un grupo de horas con
tráfico casi nulo (madrugada) y otro con tráfico alto y sostenido (día), y muy pocas
horas intermedias. No es una distribución que un logaritmo pueda arreglar, porque el
problema no es la asimetría sino que **hay dos regímenes de operación mezclados**.

Esta observación anticipa dos resultados posteriores: que K-Means encontrará
precisamente esa estructura de regímenes (punto 14), y que un modelo lineal tendrá
dificultades serias para representar la relación (punto 23).

## Ingeniería de características a partir de `date_time`

Este paso no es opcional. Las variables meteorológicas correlacionan muy débilmente con
el tráfico (`temp` apenas 0,13), mientras que el volumen medio pasa de **373 vehículos a
las 3 AM a 5.709 a las 4 PM**. La señal está en el tiempo, y viene encapsulada dentro de
`date_time`, que como texto es inutilizable para un modelo.

In [ ]:
df["hour"] = df["date_time"].dt.hour
df["weekday"] = df["date_time"].dt.dayofweek          # 0 = lunes ... 6 = domingo
df["month"] = df["date_time"].dt.month
df["is_weekend"] = (df["weekday"] >= 5).astype(int)

# Reconstruccion de la marca de feriado a las 24 horas del dia (ver punto 5)
fechas_feriado = set(df.loc[df["holiday"] != "None", "date_time"].dt.date)
df["is_holiday"] = df["date_time"].dt.date.isin(fechas_feriado).astype(int)

print("Horas marcadas como feriado ANTES de reconstruir:", int((df['holiday'] != 'None').sum()))
print("Horas marcadas como feriado DESPUES de reconstruir:", int(df['is_holiday'].sum()))
print("Dias feriado distintos:", len(fechas_feriado))
df[["date_time", "hour", "weekday", "month", "is_weekend", "is_holiday", "traffic_volume"]].head()

In [ ]:
CONTINUAS = ["temp", "rain_1h", "snow_1h", "clouds_all"]
ORDINALES = ["hour", "weekday", "month"]
NOMINALES = ["weather_main", "weather_description", "is_weekend", "is_holiday"]
PREDICTORAS = CONTINUAS + ORDINALES + NOMINALES
OBJETIVO = "traffic_volume"

print("Numéricas continuas :", CONTINUAS, f"({len(CONTINUAS)})")
print("Ordinales           :", ORDINALES, f"({len(ORDINALES)})")
print("Nominales           :", NOMINALES, f"({len(NOMINALES)})")
print("\nTotal de predictoras:", len(PREDICTORAS), "  (la ficha exige >= 6)")
print("Continuas           :", len(CONTINUAS), "  (la ficha exige >= 3)")
print("Observaciones       :", len(df), "  (la ficha exige >= 500)")

## Punto 8 · Visualizaciones exploratorias

Tres visualizaciones: matriz de correlación de Pearson, dispersión de las dos variables
más correlacionadas con el objetivo, y un violin plot del comportamiento horario.

In [ ]:
num_para_corr = CONTINUAS + ORDINALES + ["is_weekend", "is_holiday", OBJETIVO]
matriz_corr = df[num_para_corr].corr(numeric_only=True)

plt.figure(figsize=(9, 7))
mascara = np.triu(np.ones_like(matriz_corr, dtype=bool), k=1)
sns.heatmap(matriz_corr, mask=mascara, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, square=True,
            cbar_kws={"label": "Correlación de Pearson"}, linewidths=.5)
plt.title("Punto 8.1 — Matriz de correlación de Pearson", fontsize=13)
plt.xlabel("Variable"); plt.ylabel("Variable")
guardar("fig05_heatmap_correlacion.png")
plt.show()

In [ ]:
corr_obj = (matriz_corr[OBJETIVO].drop(OBJETIVO)
            .sort_values(key=np.abs, ascending=False))
top2 = corr_obj.index[:2].tolist()
print("Las 2 variables mas correlacionadas con el objetivo:", top2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
muestra = df.sample(n=4000, random_state=RANDOM_STATE)
for ax, v in zip(axes, top2):
    sns.scatterplot(data=muestra, x=v, y=OBJETIVO, alpha=.25, s=14,
                    color="#5b8ff9", edgecolor=None, ax=ax)
    ax.set_title(f"{v} vs {OBJETIVO}  (r = {corr_obj[v]:.3f})", fontsize=11)
    ax.set_xlabel(v); ax.set_ylabel("Vehículos por hora")
fig.suptitle("Punto 8.2 — Dispersión de las 2 variables más correlacionadas con el objetivo",
             fontsize=13, y=1.03)
guardar("fig06_scatter_top2.png")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.violinplot(data=df, x="hour", y=OBJETIVO, ax=axes[0],
               color="#5b8ff9", inner="quartile", linewidth=.7)
axes[0].set_title("Distribución del tráfico por hora del día", fontsize=11)
axes[0].set_xlabel("Hora del día (0–23)"); axes[0].set_ylabel("Vehículos por hora")

sns.violinplot(data=df, x="weekday", y=OBJETIVO, ax=axes[1],
               color="#5ad8a6", inner="quartile", linewidth=.7)
axes[1].set_title("Distribución del tráfico por día de la semana", fontsize=11)
axes[1].set_xlabel("Día de la semana (0 = lunes … 6 = domingo)")
axes[1].set_ylabel("Vehículos por hora")

fig.suptitle("Punto 8.3 — El patrón temporal domina el fenómeno", fontsize=13, y=1.03)
guardar("fig07_violin_hora_dia.png")
plt.show()

## Punto 9 · Top-5 correlaciones y diagnóstico de multicolinealidad

In [ ]:
top5 = corr_obj.head(5)
print("=== Top-5 variables mas correlacionadas con traffic_volume (valor absoluto) ===")
print(pd.DataFrame({"correlacion_pearson": top5.round(4),
                    "correlacion_absoluta": top5.abs().round(4)}).to_string())

In [ ]:
pred_corr = df[CONTINUAS + ORDINALES + ["is_weekend", "is_holiday"]].corr(numeric_only=True).abs()
arr = pred_corr.to_numpy().copy()          # pandas 3.0 entrega arrays de solo lectura
np.fill_diagonal(arr, 0)
pred_corr = pd.DataFrame(arr, index=pred_corr.index, columns=pred_corr.columns)

pares = (pred_corr.where(np.triu(np.ones(pred_corr.shape, dtype=bool), k=1))
         .stack().sort_values(ascending=False))

print("=== Pares de predictores mas correlacionados entre si ===")
print(pares.head(8).round(4).to_string())

UMBRAL = 0.85
criticos = pares[pares > UMBRAL]
print(f"\nPares que superan el umbral r > {UMBRAL}: {len(criticos)}")
if len(criticos):
    print(criticos.round(4).to_string())
else:
    print("Ninguno.")

### Lectura de los resultados

**Top-5 correlaciones con el objetivo.** `hour` encabeza con 0,355, seguida de
`is_weekend` (−0,213), `weekday` (−0,145), `temp` (0,139) y `clouds_all` (0,078). Dos
observaciones importantes:

1. **Todas las correlaciones son débiles.** Ninguna llega a 0,36. Pero esto **no**
   significa que no haya señal: la correlación de Pearson mide asociación *lineal*, y la
   relación entre la hora y el tráfico es marcadamente no lineal (sube hasta las 16 h y
   luego baja). Pearson subestima gravemente la importancia real de `hour`, que en el
   punto 24 resultará ser la variable dominante con lejos.
2. **El clima casi no explica el tráfico.** `rain_1h` correlaciona −0,013 y `snow_1h`
   −0,002. La gente maneja para ir a trabajar llueva o truene; el clima modula el
   volumen de forma marginal.

**Multicolinealidad.** Ningún par supera el umbral crítico de 0,85. El par más alto es
`is_weekend`–`weekday` con **0,789**, que es esperable por construcción: `is_weekend` se
deriva de `weekday`. Queda por debajo del umbral y ambas se conservan, porque codifican
matices distintos (`weekday` distingue lunes de viernes; `is_weekend` marca el corte
laboral). Aun así, se deja constancia de esta redundancia parcial, que es exactamente el
escenario donde la penalización L2 de Ridge resulta útil: reparte el peso entre
predictores correlacionados en lugar de inflar sus coeficientes.

## Punto 10 · División train/test **antes** de cualquier transformación

Esta celda es la frontera del experimento. Todo lo hecho hasta aquí fue limpieza
estructural que no usa estadísticos de los datos (deduplicar, pasar texto a minúsculas,
extraer la hora de una fecha, marcar valores físicamente imposibles). Todo lo que
**sí** aprende de los datos —imputar con la mediana, estandarizar, codificar
categorías, ajustar PCA— ocurre después del split y se ajusta **solo con `X_train`**.

In [ ]:
X = df[PREDICTORAS].copy()
y = df[OBJETIVO].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)
print(f"\nProporcion de test: {len(X_test) / len(X):.1%}")
print(f"\nMedia de y_train: {y_train.mean():.2f}  |  Media de y_test: {y_test.mean():.2f}")

> **Nota sobre `stratify`.** La ficha indica usar `stratify=y` *si es clasificación*.
> Este es un problema de regresión y `y` es continua con 6.783 valores distintos, por lo
> que la estratificación no aplica y se omite deliberadamente.

## Punto 11 · Pipeline de preprocesamiento con `ColumnTransformer`

Tres ramas, una por tipo de variable:

| Rama | Variables | Tratamiento |
|---|---|---|
| Continuas | `temp`, `rain_1h`, `snow_1h`, `clouds_all` | `SimpleImputer(mediana)` → `StandardScaler` |
| Ordinales | `hour`, `weekday`, `month` | `SimpleImputer(moda)` → `OrdinalEncoder` → `StandardScaler` |
| Nominales | `weather_main`, `weather_description`, `is_weekend`, `is_holiday` | `SimpleImputer(moda)` → `OneHotEncoder` |

`handle_unknown="ignore"` en el codificador one-hot evita que el pipeline falle si en
test aparece una categoría de clima que no estaba en train.

In [ ]:
preprocesador = ColumnTransformer(
    transformers=[
        ("continuas", Pipeline([
            ("imputador", SimpleImputer(strategy="median")),
            ("escalador", StandardScaler()),
        ]), CONTINUAS),

        ("ordinales", Pipeline([
            ("imputador", SimpleImputer(strategy="most_frequent")),
            ("codificador", OrdinalEncoder(handle_unknown="use_encoded_value",
                                           unknown_value=-1)),
            ("escalador", StandardScaler()),
        ]), ORDINALES),

        ("nominales", Pipeline([
            ("imputador", SimpleImputer(strategy="most_frequent")),
            ("codificador", OneHotEncoder(handle_unknown="ignore",
                                          sparse_output=False)),
        ]), NOMINALES),
    ],
    remainder="drop",
    verbose_feature_names_out=True,
)

# AJUSTAR solo con train, TRANSFORMAR test
X_train_prep = preprocesador.fit_transform(X_train)
X_test_prep = preprocesador.transform(X_test)

nombres_features = preprocesador.get_feature_names_out()

print("X_train_prep:", X_train_prep.shape)
print("X_test_prep :", X_test_prep.shape)
print("\nVariables generadas:", len(nombres_features))
print("  de 11 predictoras originales, el one-hot expande a", len(nombres_features))
print("\nMediana de 'temp' aprendida SOLO de train:",
      round(float(preprocesador.named_transformers_['continuas'].named_steps['imputador'].statistics_[0]), 4))

> **Sobre la ausencia de fuga de información.** El `.fit_transform()` se aplicó
> exclusivamente a `X_train`; a `X_test` solo se le aplicó `.transform()`. La mediana
> que imputa los 11 valores imposibles y las medias y desviaciones que estandarizan
> provienen únicamente del conjunto de entrenamiento. El conjunto de test permanece sin
> tocar hasta el punto 19.

## Punto 12 · PCA — varianza explicada y acumulada

In [ ]:
pca_completo = PCA(random_state=RANDOM_STATE).fit(X_train_prep)

var_exp = pca_completo.explained_variance_ratio_
var_acum = np.cumsum(var_exp)

tabla_pca = pd.DataFrame({
    "componente": [f"PC{i+1}" for i in range(len(var_exp))],
    "varianza_explicada": var_exp.round(6),
    "varianza_explicada_%": (var_exp * 100).round(3),
    "varianza_acumulada_%": (var_acum * 100).round(3),
})

n_80 = int(np.argmax(var_acum >= 0.80) + 1)
n_90 = int(np.argmax(var_acum >= 0.90) + 1)

print(f"Componentes totales: {len(var_exp)}")
print(f"Para alcanzar 80% de varianza: {n_80} componentes")
print(f"Para alcanzar 90% de varianza: {n_90} componentes")
tabla_pca.head(15)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
k = min(30, len(var_exp))

axes[0].bar(range(1, k + 1), var_exp[:k] * 100, color="#5b8ff9", alpha=.85)
axes[0].set_title("Varianza explicada por componente", fontsize=11)
axes[0].set_xlabel("Componente principal"); axes[0].set_ylabel("Varianza explicada (%)")

axes[1].plot(range(1, len(var_acum) + 1), var_acum * 100, marker="o", ms=3.5,
             color="#5b8ff9", label="Varianza acumulada")
axes[1].axhline(80, ls="--", color="#e8684a", lw=1.6, label="Umbral 80%")
axes[1].axhline(90, ls="--", color="#9270ca", lw=1.6, label="Umbral 90%")
axes[1].axvline(n_80, ls=":", color="#e8684a", lw=1.4)
axes[1].axvline(n_90, ls=":", color="#9270ca", lw=1.4)
axes[1].annotate(f"{n_80} comp. → 80%", xy=(n_80, 80), xytext=(n_80 + 6, 62),
                 arrowprops=dict(arrowstyle="->", color="#e8684a"), color="#e8684a")
axes[1].annotate(f"{n_90} comp. → 90%", xy=(n_90, 90), xytext=(n_90 + 6, 96),
                 arrowprops=dict(arrowstyle="->", color="#9270ca"), color="#9270ca")
axes[1].set_title("Scree plot — varianza acumulada", fontsize=11)
axes[1].set_xlabel("Número de componentes"); axes[1].set_ylabel("Varianza acumulada (%)")
axes[1].legend(loc="lower right")

fig.suptitle("Punto 12 — Análisis de componentes principales", fontsize=13, y=1.03)
guardar("fig08_scree_plot.png")
plt.show()

## Punto 13 · Selección de componentes, proyección PC1–PC2 y loadings

In [ ]:
N_COMPONENTES = n_80
pca = PCA(n_components=N_COMPONENTES, random_state=RANDOM_STATE)
X_train_pca = pca.fit_transform(X_train_prep)
X_test_pca = pca.transform(X_test_prep)

print(f"n_components seleccionado: {N_COMPONENTES}")
print(f"Varianza explicada acumulada: {pca.explained_variance_ratio_.sum():.4%}")
print(f"Reduccion: {X_train_prep.shape[1]} -> {N_COMPONENTES} dimensiones "
      f"({1 - N_COMPONENTES / X_train_prep.shape[1]:.1%} menos)")
print("X_train_pca:", X_train_pca.shape, " X_test_pca:", X_test_pca.shape)

In [ ]:
plt.figure(figsize=(9, 7))
sc = plt.scatter(X_train_pca[:, 0], X_train_pca[:, 1], c=y_train, cmap="viridis",
                 s=7, alpha=.45, edgecolors="none")
plt.colorbar(sc, label="traffic_volume (vehículos/hora)")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.2%} de la varianza)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.2%} de la varianza)")
plt.title("Punto 13 — Observaciones de entrenamiento en el espacio PC1–PC2\n"
          "(color = volumen de tráfico real)", fontsize=12)
guardar("fig09_pca_pc1_pc2.png")
plt.show()

In [ ]:
loadings = pd.DataFrame(
    pca.components_[:2].T,
    index=nombres_features,
    columns=["PC1", "PC2"],
)

print("=== Top-5 contribuciones a PC1 (por valor absoluto) ===")
top_pc1 = loadings["PC1"].abs().sort_values(ascending=False).head(5)
print(loadings.loc[top_pc1.index, ["PC1"]].round(4).to_string())

print("\n=== Top-5 contribuciones a PC2 (por valor absoluto) ===")
top_pc2 = loadings["PC2"].abs().sort_values(ascending=False).head(5)
print(loadings.loc[top_pc2.index, ["PC2"]].round(4).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, (comp, top) in zip(axes, [("PC1", top_pc1), ("PC2", top_pc2)]):
    vals = loadings.loc[top.index, comp].sort_values()
    colores = ["#e8684a" if v < 0 else "#5b8ff9" for v in vals]
    ax.barh(range(len(vals)), vals.values, color=colores)
    ax.set_yticks(range(len(vals)))
    ax.set_yticklabels([t.split("__")[-1] for t in vals.index], fontsize=9)
    ax.axvline(0, color="black", lw=.8)
    ax.set_title(f"Top-5 contribuciones a {comp}", fontsize=11)
    ax.set_xlabel("Loading")
fig.suptitle("Punto 13 — Variables que más contribuyen a las dos primeras componentes",
             fontsize=13, y=1.03)
guardar("fig10_loadings.png")
plt.show()

## Punto 14 · K-Means — método del codo y Silhouette Score

El Silhouette Score se calcula sobre una submuestra aleatoria de 5.000 puntos: es un
cálculo de orden O(n²) y con 32.460 observaciones resultaría inviable. La submuestra usa
la misma semilla, así que el resultado es reproducible.

In [ ]:
RANGO_K = range(2, 11)
inercias, silhouettes = [], []

rng = np.random.RandomState(RANDOM_STATE)
idx_sil = rng.choice(len(X_train_pca), size=min(5000, len(X_train_pca)), replace=False)

for k in RANGO_K:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    etiquetas = km.fit_predict(X_train_pca)
    inercias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_train_pca[idx_sil], etiquetas[idx_sil]))

tabla_k = pd.DataFrame({
    "K": list(RANGO_K),
    "inercia": np.round(inercias, 1),
    "silhouette": np.round(silhouettes, 4),
    "caida_inercia_%": [np.nan] + list(
        np.round(-np.diff(inercias) / np.array(inercias[:-1]) * 100, 2)),
})
tabla_k

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(list(RANGO_K), inercias, marker="o", color="#5b8ff9", lw=1.8)
axes[0].set_title("Método del codo — inercia (WCSS)", fontsize=11)
axes[0].set_xlabel("Número de clusters (K)"); axes[0].set_ylabel("Inercia")
axes[0].axvline(3, ls="--", color="#e8684a", lw=1.5, label="K = 3 seleccionado")
axes[0].legend()

axes[1].plot(list(RANGO_K), silhouettes, marker="o", color="#5ad8a6", lw=1.8)
axes[1].set_title("Silhouette Score por K", fontsize=11)
axes[1].set_xlabel("Número de clusters (K)"); axes[1].set_ylabel("Silhouette Score")
axes[1].axvline(3, ls="--", color="#e8684a", lw=1.5, label="K = 3 seleccionado")
axes[1].legend()

fig.suptitle("Punto 14 — Selección del número de clusters", fontsize=13, y=1.03)
guardar("fig11_codo_silhouette.png")
plt.show()

print(f"Mejor silhouette absoluto: K = {list(RANGO_K)[int(np.argmax(silhouettes))]} "
      f"(score {max(silhouettes):.4f})")
print(f"Rango completo de silhouette: {min(silhouettes):.4f} a {max(silhouettes):.4f}")

### Selección del K óptimo — justificación

**Se selecciona K = 3**, y conviene explicar por qué no se elige el K con mayor
Silhouette absoluto.

El Silhouette crece de forma casi monótona hasta K = 10, pero **todo el rango se mueve
entre 0,208 y 0,239**. Son valores bajos: un Silhouette cercano a 0,2 indica que la
separación entre grupos es débil y que las fronteras son difusas. La diferencia entre el
mejor y el peor K es de apenas 0,03, muy por debajo de lo que justificaría preferir una
solución de 10 grupos. Perseguir ese máximo sería sobreinterpretar ruido.

El método del codo aporta el criterio que falta: la caída porcentual de inercia se
desacelera notoriamente a partir de K = 3–4. Y K = 3 presenta además un **máximo local**
de Silhouette (0,2187, por sobre 0,2178 de K = 2 y 0,2083 de K = 4).

El argumento decisivo es de interpretabilidad. La curtosis negativa del punto 7 ya había
anticipado una estructura bimodal, y K = 3 produce grupos que se corresponden con
regímenes de operación reconocibles de una autopista —madrugada, valle diurno y horas
punta—, como se verifica en el punto 15. Diez clusters serían estadísticamente
marginales e imposibles de traducir a una recomendación operativa.

**Advertencia honesta:** un Silhouette de 0,22 significa que los datos no presentan
grupos naturalmente separados. La variable subyacente (la hora) es continua y cíclica,
así que K-Means impone cortes sobre un continuo en vez de descubrir agrupaciones
discretas. Los clusters son útiles como descripción, no como evidencia de que existan
tres tipos de horas esencialmente distintos.

## Punto 15 · Asignación de clusters y perfil por grupo

In [ ]:
K_OPTIMO = 3
kmeans = KMeans(n_clusters=K_OPTIMO, random_state=RANDOM_STATE, n_init=10)
clusters_train = kmeans.fit_predict(X_train_pca)

print("Distribución de observaciones por cluster:")
print(pd.Series(clusters_train).value_counts().sort_index().to_string())

In [ ]:
plt.figure(figsize=(9, 7))
paleta = ["#5b8ff9", "#5ad8a6", "#e8684a"]
for c in range(K_OPTIMO):
    m = clusters_train == c
    plt.scatter(X_train_pca[m, 0], X_train_pca[m, 1], s=7, alpha=.4,
                color=paleta[c], label=f"Cluster {c}  (n = {m.sum():,})",
                edgecolors="none")
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
            marker="X", s=280, c="black", edgecolors="white", linewidths=1.6,
            label="Centroides", zorder=5)
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.2%} de la varianza)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.2%} de la varianza)")
plt.title(f"Punto 15 — Clusters de K-Means (K = {K_OPTIMO}) en el espacio PCA", fontsize=12)
plt.legend(loc="best", framealpha=.92)
guardar("fig12_clusters_pca.png")
plt.show()

In [ ]:
perfil_base = X_train.copy()
perfil_base["cluster"] = clusters_train
perfil_base["traffic_volume"] = y_train.values

VARS_PERFIL = ["hour", "weekday", "month", "temp", "rain_1h", "snow_1h",
               "clouds_all", "is_weekend", "is_holiday", "traffic_volume"]

perfil = perfil_base.groupby("cluster")[VARS_PERFIL].mean().round(3)
perfil["n_observaciones"] = perfil_base.groupby("cluster").size()
print("=== Perfil de clusters: media de cada variable por grupo ===")
perfil

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

sns.boxplot(data=perfil_base, x="cluster", y="hour", ax=axes[0], palette=paleta, hue="cluster", legend=False)
axes[0].set_title("Hora del día por cluster", fontsize=11)
axes[0].set_xlabel("Cluster"); axes[0].set_ylabel("Hora (0–23)")

sns.boxplot(data=perfil_base, x="cluster", y="traffic_volume", ax=axes[1], palette=paleta, hue="cluster", legend=False)
axes[1].set_title("Volumen de tráfico por cluster", fontsize=11)
axes[1].set_xlabel("Cluster"); axes[1].set_ylabel("Vehículos por hora")

sns.boxplot(data=perfil_base, x="cluster", y="temp", ax=axes[2], palette=paleta, hue="cluster", legend=False)
axes[2].set_title("Temperatura por cluster", fontsize=11)
axes[2].set_xlabel("Cluster"); axes[2].set_ylabel("Temperatura (°C)")

fig.suptitle("Punto 15 — Perfil de los clusters encontrados", fontsize=13, y=1.04)
guardar("fig13_perfil_clusters.png")
plt.show()

## Puntos 16 a 18 · Entrenamiento de modelos con validación cruzada

Se entrenan tres estimadores, todos con `random_state=42`:

1. **Baseline** (`DummyRegressor`) — predice siempre la media. No lo pide la ficha, pero
   sin un piso de comparación una métrica no significa nada.
2. **Ridge** — modelo penalizado (L2), con `alpha` ajustado por `GridSearchCV`.
3. **Random Forest** — modelo basado en árboles, con `n_estimators`, `max_depth` y
   `min_samples_split` ajustados por `GridSearchCV`.

Ambas búsquedas usan `cv=5` y se ajustan **solo sobre datos de entrenamiento**.

In [ ]:
# ---------------------------- BASELINE ----------------------------
t0 = time.time()
baseline = DummyRegressor(strategy="mean").fit(X_train_prep, y_train)
t_baseline = time.time() - t0
print(f"Baseline entrenado en {t_baseline:.4f} s")
print(f"Prediccion constante: {baseline.constant_[0][0]:.2f} vehiculos/hora")

In [ ]:
# --------------------- PUNTO 17: RIDGE + GRIDSEARCH ---------------------
grid_ridge = {"alpha": [0.001, 0.01, 0.1, 1, 10, 100]}

t0 = time.time()
busqueda_ridge = GridSearchCV(
    Ridge(random_state=RANDOM_STATE),
    param_grid=grid_ridge,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
)
busqueda_ridge.fit(X_train_prep, y_train)
t_ridge = time.time() - t0

ridge = busqueda_ridge.best_estimator_

print(f"Tiempo de busqueda + entrenamiento: {t_ridge:.2f} s")
print(f"Mejores hiperparametros (.best_params_): {busqueda_ridge.best_params_}")
print(f"Mejor RMSE de validacion cruzada: {-busqueda_ridge.best_score_:.4f}")
print("\nResultados por valor de alpha:")
print(pd.DataFrame({
    "alpha": grid_ridge["alpha"],
    "RMSE_cv": np.round(-busqueda_ridge.cv_results_["mean_test_score"], 4),
    "desv_std": np.round(busqueda_ridge.cv_results_["std_test_score"], 4),
}).to_string(index=False))

In [ ]:
# --------------------- PUNTO 18: RANDOM FOREST + GRIDSEARCH ---------------------
grid_rf = {
    "n_estimators": [100, 300],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 10],
}

t0 = time.time()
busqueda_rf = GridSearchCV(
    RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    param_grid=grid_rf,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    verbose=1,
)
busqueda_rf.fit(X_train_prep, y_train)
t_rf = time.time() - t0

rf = busqueda_rf.best_estimator_

print(f"\nTiempo de busqueda + entrenamiento: {t_rf:.2f} s")
print(f"Combinaciones evaluadas: {len(busqueda_rf.cv_results_['params'])} x 5 folds")
print(f"Mejores hiperparametros (.best_params_): {busqueda_rf.best_params_}")
print(f"Mejor RMSE de validacion cruzada: {-busqueda_rf.best_score_:.4f}")

In [ ]:
resultados_rf = pd.DataFrame(busqueda_rf.cv_results_)[
    ["param_n_estimators", "param_max_depth", "param_min_samples_split",
     "mean_test_score", "std_test_score"]
].copy()
resultados_rf["RMSE_cv"] = (-resultados_rf["mean_test_score"]).round(4)
resultados_rf = resultados_rf.drop(columns=["mean_test_score"]).sort_values("RMSE_cv")
print("=== Resultados de la busqueda en grilla del Random Forest ===")
resultados_rf.head(12)

## Punto 19 · Tiempos de entrenamiento y predicción sobre el conjunto de test

Primera y única vez que se usa `X_test_prep`, exclusivamente para predecir.

In [ ]:
def medir_reentrenamiento(estimador):
    '''Reentrena con los mejores hiperparametros y mide el tiempo limpio.'''
    from sklearn.base import clone
    m = clone(estimador)
    t0 = time.time()
    m.fit(X_train_prep, y_train)
    return m, time.time() - t0

baseline, t_fit_baseline = medir_reentrenamiento(baseline)
ridge, t_fit_ridge = medir_reentrenamiento(ridge)
rf, t_fit_rf = medir_reentrenamiento(rf)

modelos = {
    "Baseline (media)": (baseline, t_fit_baseline),
    "Ridge (L2)": (ridge, t_fit_ridge),
    "Random Forest": (rf, t_fit_rf),
}

predicciones, tiempos_inferencia = {}, {}
for nombre, (modelo, _) in modelos.items():
    t0 = time.time()
    predicciones[nombre] = modelo.predict(X_test_prep)
    tiempos_inferencia[nombre] = time.time() - t0
    print(f"{nombre:20s} entrenamiento {modelos[nombre][1]:8.4f} s | "
          f"inferencia {tiempos_inferencia[nombre]:.4f} s")

## Punto 20 · Métricas sobre el conjunto de test

RMSE, MAE, R² y MAPE, con cuatro decimales.

In [ ]:
def mape(y_real, y_pred):
    '''MAPE excluyendo valores reales muy cercanos a cero.'''
    y_real = np.asarray(y_real, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.abs(y_real) > 1e-9
    return float(np.mean(np.abs((y_real[mask] - y_pred[mask]) / y_real[mask])) * 100)

def calcular_metricas(y_real, y_pred):
    return {
        "RMSE": float(np.sqrt(mean_squared_error(y_real, y_pred))),
        "MAE": float(mean_absolute_error(y_real, y_pred)),
        "R2": float(r2_score(y_real, y_pred)),
        "MAPE_%": mape(y_real, y_pred),
    }

for nombre, pred in predicciones.items():
    m = calcular_metricas(y_test, pred)
    print(f"--- {nombre} ---")
    for k_, v_ in m.items():
        print(f"    {k_:8s}: {v_:.4f}")

> **Advertencia sobre el MAPE.** Los valores que arroja son enormes y **no deben
> interpretarse como error porcentual real**. `traffic_volume` alcanza valores mínimos de
> 2 vehículos por hora en la madrugada; equivocarse por 300 vehículos sobre un valor real
> de 2 produce un error de 15.000%, y ese puñado de horas domina el promedio. El MAPE es
> una métrica inadecuada para objetivos con valores cercanos a cero. Se reporta porque la
> ficha lo exige, pero **la métrica principal de este trabajo es el RMSE**, expresado en
> vehículos por hora y directamente interpretable.

## Punto 21 · Tabla comparativa de modelos

In [ ]:
filas = []
for nombre, (modelo, t_fit) in modelos.items():
    m_test = calcular_metricas(y_test, predicciones[nombre])
    m_train = calcular_metricas(y_train, modelo.predict(X_train_prep))
    filas.append({
        "modelo": nombre,
        "RMSE_test": round(m_test["RMSE"], 4),
        "MAE_test": round(m_test["MAE"], 4),
        "R2_test": round(m_test["R2"], 4),
        "MAPE_%_test": round(m_test["MAPE_%"], 4),
        "R2_train": round(m_train["R2"], 4),
        "RMSE_train": round(m_train["RMSE"], 4),
        "brecha_R2": round(m_train["R2"] - m_test["R2"], 4),
        "tiempo_entrenamiento_s": round(t_fit, 4),
        "tiempo_inferencia_s": round(tiempos_inferencia[nombre], 4),
    })

comparativa = pd.DataFrame(filas).sort_values("RMSE_test").reset_index(drop=True)
comparativa.to_csv("resultados_modelos.csv", index=False)
print("Tabla exportada a resultados_modelos.csv")
comparativa

## Punto 22 · Valores reales vs predichos e histograma de residuales

In [ ]:
mejor_nombre = comparativa.loc[0, "modelo"]
mejor_modelo = modelos[mejor_nombre][0]
mejor_pred = predicciones[mejor_nombre]
print("Mejor modelo por RMSE de test:", mejor_nombre)

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

for nombre, pred in predicciones.items():
    if nombre == "Baseline (media)":
        continue
    axes[0].scatter(y_test, pred, s=6, alpha=.25, label=nombre, edgecolors="none")
lim = [0, y_test.max() * 1.02]
axes[0].plot(lim, lim, "--", color="black", lw=1.4, label="Predicción perfecta")
axes[0].set_xlim(lim); axes[0].set_ylim(lim)
axes[0].set_xlabel("Valor real (vehículos/hora)")
axes[0].set_ylabel("Valor predicho (vehículos/hora)")
axes[0].set_title("Valores reales vs predichos sobre test", fontsize=11)
axes[0].legend(loc="upper left", markerscale=3)

residuales = y_test - mejor_pred
sns.histplot(residuales, bins=70, kde=True, color="#5b8ff9", ax=axes[1])
axes[1].axvline(0, ls="--", color="black", lw=1.4)
axes[1].set_title(f"Residuales de {mejor_nombre}\n"
                  f"media = {residuales.mean():.2f} | desv. est. = {residuales.std():.2f}",
                  fontsize=11)
axes[1].set_xlabel("Residual (real − predicho)"); axes[1].set_ylabel("Frecuencia")

fig.suptitle("Punto 22 — Diagnóstico del ajuste", fontsize=13, y=1.03)
guardar("fig14_real_vs_predicho_residuales.png")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))

axes[0].scatter(mejor_pred, residuales, s=6, alpha=.25, color="#5b8ff9", edgecolors="none")
axes[0].axhline(0, ls="--", color="black", lw=1.4)
axes[0].set_xlabel("Valor predicho"); axes[0].set_ylabel("Residual")
axes[0].set_title("Residuales vs valores predichos", fontsize=11)

stats.probplot(residuales, dist="norm", plot=axes[1])
axes[1].set_title("Gráfico Q-Q de los residuales", fontsize=11)
axes[1].get_lines()[0].set_markersize(2.5)
axes[1].get_lines()[0].set_alpha(.35)

fig.suptitle(f"Punto 22 — Diagnóstico complementario de residuales ({mejor_nombre})",
             fontsize=13, y=1.04)
guardar("fig15_diagnostico_residuales.png")
plt.show()

## Punto 24 · Importancia de variables del mejor modelo

In [ ]:
importancias = pd.Series(mejor_modelo.feature_importances_,
                         index=nombres_features).sort_values(ascending=False)

top15 = importancias.head(15)
plt.figure(figsize=(10, 7))
etiquetas = [t.split("__")[-1] for t in top15.index][::-1]
plt.barh(range(len(top15)), top15.values[::-1], color="#5b8ff9")
plt.yticks(range(len(top15)), etiquetas, fontsize=9)
plt.xlabel("Importancia (reducción media de impureza)")
plt.ylabel("Variable")
plt.title(f"Punto 24 — Top-15 variables más importantes ({mejor_nombre})", fontsize=12)
guardar("fig16_importancia_variables.png")
plt.show()

print("=== Top-10 variables por importancia ===")
print(importancias.head(10).round(4).to_string())
print(f"\nLas 5 primeras concentran el {importancias.head(5).sum():.2%} de la importancia total")

In [ ]:
comparacion_final = pd.DataFrame({
    "importancia_RF": importancias.head(8).round(4).values,
}, index=[t.split("__")[-1] for t in importancias.head(8).index])

corr_map = {v: corr_obj.get(v, np.nan) for v in comparacion_final.index}
comparacion_final["correlacion_Pearson_EDA"] = [
    round(corr_map[v], 4) if pd.notna(corr_map.get(v, np.nan)) else "—"
    for v in comparacion_final.index
]
print("=== Importancia del modelo vs correlacion lineal del EDA ===")
comparacion_final

## Punto 25 · Las 10 observaciones con mayor error

In [ ]:
analisis_err = X_test.copy()
analisis_err["real"] = y_test.values
analisis_err["predicho"] = mejor_pred
analisis_err["residual"] = analisis_err["real"] - analisis_err["predicho"]
analisis_err["error_abs"] = analisis_err["residual"].abs()

peores = analisis_err.nlargest(10, "error_abs")
print("=== 10 observaciones con mayor error absoluto ===")
print(peores[["hour", "weekday", "month", "is_weekend", "is_holiday", "temp",
              "weather_main", "real", "predicho", "residual"]].round(2).to_string())

In [ ]:
print("=== Comparacion: 10 peores errores vs el resto del conjunto de test ===")
resto = analisis_err.drop(peores.index)
comparativa_err = pd.DataFrame({
    "10_peores": {
        "error_abs_medio": peores["error_abs"].mean().round(1),
        "trafico_real_medio": peores["real"].mean().round(1),
        "hora_media": peores["hour"].mean().round(1),
        "%_fin_de_semana": round(peores["is_weekend"].mean() * 100, 1),
        "%_feriado": round(peores["is_holiday"].mean() * 100, 1),
        "temp_media": peores["temp"].mean().round(1),
    },
    "resto_del_test": {
        "error_abs_medio": resto["error_abs"].mean().round(1),
        "trafico_real_medio": resto["real"].mean().round(1),
        "hora_media": resto["hour"].mean().round(1),
        "%_fin_de_semana": round(resto["is_weekend"].mean() * 100, 1),
        "%_feriado": round(resto["is_holiday"].mean() * 100, 1),
        "temp_media": resto["temp"].mean().round(1),
    },
})
print(comparativa_err.to_string())

print("\nDistribucion horaria de los 10 peores errores:")
print(peores["hour"].value_counts().sort_index().to_string())

## Anexo · ¿Por qué Ridge rinde tan por debajo del Random Forest?

El punto 11 prescribe tratar `hour` con `OrdinalEncoder`, lo que la convierte en un
número del 0 al 23 y obliga al modelo lineal a asignarle **un solo coeficiente**. Pero
el tráfico sube de las 3 AM a las 4 PM y después baja: la relación no es monótona, así
que ningún coeficiente único puede representarla.

Este experimento aísla ese efecto: mismo modelo, mismos datos, misma semilla; lo único
que cambia es cómo se codifica la hora.

In [ ]:
def preprocesador_alternativo(ordinales, nominales):
    return ColumnTransformer([
        ("continuas", Pipeline([("i", SimpleImputer(strategy="median")),
                                ("s", StandardScaler())]), CONTINUAS),
        ("ordinales", Pipeline([("i", SimpleImputer(strategy="most_frequent")),
                                ("c", OrdinalEncoder(handle_unknown="use_encoded_value",
                                                     unknown_value=-1)),
                                ("s", StandardScaler())]), ordinales),
        ("nominales", Pipeline([("i", SimpleImputer(strategy="most_frequent")),
                                ("c", OneHotEncoder(handle_unknown="ignore",
                                                    sparse_output=False))]), nominales),
    ])

# B: 'hour' tratada como nominal (one-hot), todo lo demas identico
prep_b = preprocesador_alternativo(["weekday", "month"], NOMINALES + ["hour"])
Xtr_b = prep_b.fit_transform(X_train)
Xte_b = prep_b.transform(X_test)

ridge_b = Ridge(alpha=ridge.alpha, random_state=RANDOM_STATE).fit(Xtr_b, y_train)
r2_a = r2_score(y_test, predicciones["Ridge (L2)"])
r2_b = r2_score(y_test, ridge_b.predict(Xte_b))
rmse_a = np.sqrt(mean_squared_error(y_test, predicciones["Ridge (L2)"]))
rmse_b = np.sqrt(mean_squared_error(y_test, ridge_b.predict(Xte_b)))

print("Ridge con 'hour' ORDINAL (lo prescrito por la ficha):")
print(f"    R2 = {r2_a:.4f}   RMSE = {rmse_a:.4f}   [{Xtr_b.shape[1] - 23} variables]")
print("Ridge con 'hour' ONE-HOT (solo cambia la codificacion):")
print(f"    R2 = {r2_b:.4f}   RMSE = {rmse_b:.4f}   [{Xtr_b.shape[1]} variables]")
print(f"\nGanancia en R2: {r2_b - r2_a:+.4f}   |   Reduccion de RMSE: {rmse_a - rmse_b:.1f} vehiculos/hora")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))

medias_hora = df.groupby("hour")["traffic_volume"].mean()
axes[0].plot(medias_hora.index, medias_hora.values, marker="o", color="#5b8ff9", lw=2)
axes[0].set_title("Tráfico medio por hora: la relación no es monótona", fontsize=11)
axes[0].set_xlabel("Hora del día"); axes[0].set_ylabel("Vehículos por hora (media)")
axes[0].annotate("sube hasta las 16 h", xy=(16, medias_hora[16]), xytext=(8, 5900),
                 arrowprops=dict(arrowstyle="->", color="#e8684a"), color="#e8684a")
axes[0].annotate("y vuelve a bajar", xy=(21, medias_hora[21]), xytext=(18, 1200),
                 arrowprops=dict(arrowstyle="->", color="#e8684a"), color="#e8684a")

nombres_b = ["Ridge\n(hour ordinal)", "Ridge\n(hour one-hot)", "Random Forest"]
valores_b = [r2_a, r2_b, comparativa.loc[comparativa["modelo"] == "Random Forest", "R2_test"].iloc[0]]
barras = axes[1].bar(nombres_b, valores_b, color=["#e8684a", "#f6bd16", "#5ad8a6"])
axes[1].bar_label(barras, fmt="%.4f", fontsize=10)
axes[1].set_ylim(0, 1.05)
axes[1].set_title("R² sobre test según codificación de la hora", fontsize=11)
axes[1].set_ylabel("R² en test"); axes[1].set_xlabel("Configuración del modelo")

fig.suptitle("Anexo — El costo de codificar como ordinal una variable cíclica",
             fontsize=13, y=1.04)
guardar("fig17_experimento_codificacion.png")
plt.show()

<!-- MARCADOR_PUNTO_23 -->

<!-- MARCADOR_PUNTO_26 -->